# DE5 StarRocks Starter

PySpark 커널 안에서 StarRocks를 같이 조회하는 실습 노트북입니다.

- Spark/Paimon/Iceberg 조회: `spark.sql(...)`
- StarRocks OLAP 조회: `starrocks_sql(...)` 또는 `sr_sql(...)`

커널은 반드시 **PySpark (DE5 Lakehouse)** 를 선택하세요.

## 1. StarRocks catalog 확인

StarRocks는 MySQL protocol로 접속합니다. Jupyter 컨테이너 안에서는 `starrocks-fe:9030`으로 접근합니다.

In [1]:
starrocks_sql("SHOW CATALOGS")

,Catalog,Type,Comment
0,default_catalog,Internal,An internal catalog contains this cluster's se...
1,iceberg_olist,Iceberg,None
2,paimon_olist,Paimon,None


In [2]:
starrocks_sql("SHOW DATABASES FROM paimon_olist")

,Database
0,bronze
1,default
2,information_schema


## 2. Paimon external catalog 조회

StarRocks가 Paimon 데이터를 내부 테이블로 복사하는 것이 아니라, Paimon external catalog로 Lakehouse table을 직접 조회합니다.

In [3]:
starrocks_sql("""
SELECT 'ux_events_bronze' AS table_name, COUNT(*) AS row_count
FROM paimon_olist.bronze.ux_events_bronze
UNION ALL
SELECT 'review_current' AS table_name, COUNT(*) AS row_count
FROM paimon_olist.bronze.review_current
UNION ALL
SELECT 'order_current' AS table_name, COUNT(*) AS row_count
FROM paimon_olist.bronze.order_current
""")

,table_name,row_count
0,review_current,1971
1,ux_events_bronze,16693
2,order_current,2000


In [4]:
starrocks_sql("""
SELECT
  category_code,
  COUNT(*) AS review_count,
  SUM(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) AS negative_reviews,
  ROUND(100.0 * SUM(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) / COUNT(*), 2) AS negative_rate_pct
FROM paimon_olist.bronze.review_current
WHERE category_code IS NOT NULL
GROUP BY category_code
HAVING COUNT(*) >= 10
ORDER BY negative_rate_pct DESC, review_count DESC
LIMIT 15
""")

,category_code,review_count,negative_reviews,negative_rate_pct
0,bebes,46,16,34.78
1,moveis_escritorio,18,5,27.78
2,pet_shop,28,7,25.00
3,cama_mesa_banho,140,34,24.29
4,instrumentos_musicais,13,3,23.08
5,papelaria,66,15,22.73
6,ferramentas_jardim,84,19,22.62
7,brinquedos,200,44,22.00
8,cool_stuff,96,20,20.83
9,utilidades_domesticas,85,16,18.82


## 3. Iceberg external catalog 확인

Spark batch가 만든 Iceberg Analytics table도 StarRocks Iceberg external catalog로 조회할 수 있습니다.

In [5]:
starrocks_sql("SHOW DATABASES FROM iceberg_olist")

,Database
0,analytics
1,bronze
2,information_schema


In [6]:
starrocks_sql("SHOW TABLES FROM iceberg_olist.analytics")

,Tables_in_analytics
0,github_event_type_daily
1,github_events_clean
2,olist_category_daily
3,olist_event_type_daily
4,olist_funnel_daily
5,olist_order_current
6,olist_review_current
7,olist_review_sentiment_by_category
8,olist_ux_events_clean


## 4. 같은 노트북에서 Spark와 StarRocks 같이 보기

`spark.sql`은 Spark가 직접 Lakehouse table을 읽는 경로이고, `starrocks_sql`은 StarRocks serving layer를 통해 조회하는 경로입니다.

In [7]:
spark.sql("SELECT COUNT(*) AS ux_rows FROM paimon_lake.bronze.ux_events_bronze").show()
starrocks_sql("SELECT COUNT(*) AS ux_rows FROM paimon_olist.bronze.ux_events_bronze")

26/06/07 10:50:53 WARN HadoopUtils: Could not find Hadoop configuration via any of the supported methods


+-------+
|ux_rows|
+-------+
|  16693|
+-------+



,ux_rows
0,16693
